In [1]:
# THIS NOTEBOOK IS USED TO CREATE A DRAFT FOR THE PREPROCESSING OF EVERY CSV 

## EDA Checklist

### Drop Unnecessary Columns
- Remove irrelevant columns from dataset.
### Handle Missing Values

**Drop rows if missing:**
- `wpm`
- `accuracy`
- `timestamp` → **Must exist**

**Fill missing values:**
- `consistency` → fill with **median**
- `language` → fill with `"unknown"`

### Change Data Types
- Convert columns to correct data types  
  - Example:
    - `timestamp` → datetime
    - numeric columns → int/float
    - categorical columns → string/category
### Rename Columns as per DB Schema
- Rename dataframe columns according to database schema requirements.

### Remove Impossible Values

Remove rows where:
- `wpm < 0`
- `accuracy > 100`
- `accuracy < 0`
- `consistency > 100`

### Sort Data by Timestamp
```python
df.sort_values("test_time")
```
### Create Time-Based Features

**Days since first test:**
```python
days = test_time - first_test_time
```
###  Remove Outliers

Remove rows where:
- `wpm <= 0`
- `wpm > 250` → humanly unrealistic
- `accuracy <= 0`
- `accuracy > 100`

### Daily Average WPM

**Group by day:**
- Compute average WPM per day

**Output format:**
```
date → avg_wpm
```


### database schema

    id SERIAL PRIMARY KEY,
     user_id INT REFERENCES users(id),

     test_time TIMESTAMP,

     wpm FLOAT,
     raw_wpm FLOAT,
     accuracy FLOAT,
     consistency FLOAT,

     correct_chars INT,
     incorrect_chars INT,
     extra_chars INT,
     missed_chars INT,

     test_duration FLOAT,
     mode TEXT,
     mode2 INT,
     quote_length INT,
     language TEXT,
     difficulty TEXT,

     is_pb BOOLEAN,
     punctuation BOOLEAN,
     numbers BOOLEAN

In [5]:
import pandas as pd
import numpy as np


In [3]:
def load_csv():
    data=pd.read_csv("/home/akshajtiwari/Downloads/results.csv")
    df=pd.DataFrame(data)
    return df
df=load_csv()
def drop_columns(df):
    df.drop(columns=['afkDuration','restartCount','incompleteTestSeconds', 'funbox', 'lazyMode', 'blindMode',
       'bailedOut', 'tags'],inplace=True)
    return dfuser_id
df=drop_columns(df)
def remove_nan(df):
    columns_drop=['wpm','acc']
    df = df.dropna(subset=columns_drop)
    # consistency median, language unknown
    median=df['consistency'].median()
    df['consistency']=df['consistency'].fillna(median)
    df['language']=df['language'].fillna("unknown")
    return df
df=remove_nan(df)
def change_dtypes(df):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')    
    df['isPb']=df['isPb'].astype('bool')
    df['mode2']=df['mode2'].replace("custom",-1).astype(int) # -1 represents custom time durations 
    return df
df=change_dtypes(df)
def split_char_stats(df):
    char_split = df["charStats"].str.split(";", expand=True)
    df["correct_chars"] = char_split[0].astype(int)
    df["incorrect_chars"] = char_split[1].astype(int)
    df["extra_chars"] = char_split[2].astype(int)
    df["missed_chars"] = char_split[3].astype(int)
    # Drop original column
    df = df.drop(columns=["charStats"])
    return df
df=split_char_stats(df)
def rename_columns(df):
    df.rename(columns={
        "_id": "id_data",
        "timestamp": "test_time",
        "rawWpm": "raw_wpm",
        "acc": "accuracy",
        "isPb": "is_pb",
        "quoteLength": "quote_length",
        "testDuration": "test_duration"
    }, inplace=True)
    return df
df=rename_columns(df)
def outliers(df):
    df = df[(df['wpm'] > 0) & (df['wpm']<250)]
    df = df[(df['accuracy'] > 0) & (df['accuracy'] <= 100)]
    # df = df[df['consistency'] <= 100]
    return df
df=outliers(df)
def sort_by_time(df):
    df=df.sort_values(by=["test_time"])
    df = df.reset_index(drop=True) # reset index to start from 0 instead of last test index
    
    #df['days'] = (df['timestamp'] - df['timestamp'].iloc[0]).dt.days
#     daily_avg = df.groupby('days').agg({
#     'wpm':'mean',
#     'acc':'mean',
#     'rawWpm':'mean'
# }).reset_index()
    return df
df=sort_by_time(df)


In [4]:
df['test_time']

0     2026-02-15 07:22:49
1     2026-02-15 07:20:52
2     2026-02-14 18:48:07
3     2026-02-14 04:17:12
4     2026-02-13 05:58:11
              ...        
995   2024-08-03 03:50:13
996   2024-08-02 15:53:31
997   2024-08-01 04:49:40
998   2024-08-01 04:40:16
999   2024-08-01 04:39:39
Name: test_time, Length: 1000, dtype: datetime64[ms]

In [8]:
data.head()


,_id,isPb,wpm,acc,rawWpm,consistency,charStats,mode,mode2,quoteLength,...,punctuation,numbers,language,funbox,difficulty,lazyMode,blindMode,bailedOut,tags,timestamp
0,69917449d5fe59bad8def2e2,NaN,90.41,95.13,95.81,73.89,452;7;0;3,time,60,-1,...,False,False,english,NaN,normal,False,False,False,NaN,1771140169000
1,699173d4d5fe59bad8deef73,NaN,97.21,95.69,97.21,75.00,486;0;0;0,time,60,-1,...,False,False,english,NaN,normal,False,False,False,NaN,1771140052000
2,6990c367d5fe59bad8dac5e3,True,104.01,98.13,105.61,76.54,520;2;0;0,time,60,-1,...,False,False,english,NaN,normal,False,False,False,NaN,1771094887000
3,698ff748d5fe59bad8d3fade,NaN,92.61,93.60,100.21,73.22,463;12;1;3,time,60,-1,...,False,False,english,NaN,normal,False,False,False,NaN,1771042632000
4,698ebd73d5fe59bad8c8163d,NaN,92.61,93.63,100.61,74.10,463;12;0;1,time,60,-1,...,False,False,english,NaN,normal,False,False,False,NaN,1770962291000
